In [1]:
# WEBCAM DISTANCE ESTIMATION
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
# from ultralytics import YOLO

# Depth Anything V2
import sys
from pathlib import Path
repo_path = Path.cwd() / "Depth-Anything-V2"
sys.path.append(str(repo_path))
from depth_anything_v2.dpt import DepthAnythingV2


W0506 17:29:14.924000 18116 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
# Load Depth Estimation Model (Depth-Anything-V2)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},       # Small
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},     # Base
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},  # Large
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]} # Giant
}

# Choose Version
encoder = 'vits' # or 'vits', 'vitb', 'vitg'
ckpt_path = repo_path / "checkpoints" / f"depth_anything_v2_{encoder}.pth"

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
model = model.to(DEVICE).eval()

In [3]:
# Open the camera
cam = cv2.VideoCapture(0)

# Make sure the camera is open
if not cam.isOpened():
    print("The camera isn't opening! :(")
    exit()
    quit()

# Get Frame Dimensions
dims = np.array([int(cam.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))])
print("Frame Dimensions: [%i, %i]" % (dims[0], dims[1]))

Frame Dimensions: [640, 480]


In [4]:
print('Press "q" or ESC key to quit the application')

fail_count = 0 # failed frame acquisition count
fail_limit = 10 # maximum number of failed frame acquisitions before the program exits

# Camera object search & display loop
while True:
    ret, frame = cam.read()

    # If the frame was successfully captured...
    if ret:

        # Estimate Depth
        depth = model.infer_image(frame) # HxW raw depth map in numpy
        # Display Depth
        depth_01 = (depth-np.min(depth))/(np.max(depth)-np.min(depth))
        cv2.imshow("Depth (0-1 Normalized)", depth_01)
        
    else:
        fail_count += 1
        print("Failed to capture frame from camera")
        if fail_count > fail_limit:
            print("Failed to capture frame from camera %i times, exiting program..." % fail_limit)
            break

    # Press 'q' to exit the loop
    if (cv2.waitKey(1) == ord('q')) or (cv2.waitKey(1) == 27):
        break

# Release the camera capture object
cam.release()
# Close Windows
cv2.destroyAllWindows()

Press "q" or ESC key to quit the application
